# Notebook 05: Vector Store Indexing & Retrieval

This notebook builds FAISS indices from the embeddings generated in Notebook 04 and
implements the retrieval interface. We support three retrieval modes: text-to-text,
text-to-image (cross-modal via CLIP), and hybrid (weighted combination).

**Goals:**
1. Build FAISS flat indices for text and image embeddings
2. Implement retrieval functions (top-K with metadata)
3. Implement hybrid retrieval (text + image score fusion)
4. Add BM25 sparse retrieval as a baseline
5. Test retrieval on sample NExT-QA questions

**Inputs:** `data/processed/embeddings/` from Notebook 04

**Outputs:**
- Retrieval functions ready for evaluation in Notebook 06
- `data/processed/indices/` -- serialized FAISS indices

In [1]:
import os
os.environ['HF_HUB_DISABLE_SSL_VERIFY'] = '1'
os.environ['REQUESTS_CA_BUNDLE'] = ''
os.environ['CURL_CA_BUNDLE'] = ''
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import httpx
_orig_client_init = httpx.Client.__init__
def _patched_client_init(self, *args, **kwargs):
    kwargs['verify'] = False
    _orig_client_init(self, *args, **kwargs)
httpx.Client.__init__ = _patched_client_init
_orig_async_init = httpx.AsyncClient.__init__
def _patched_async_init(self, *args, **kwargs):
    kwargs['verify'] = False
    _orig_async_init(self, *args, **kwargs)
httpx.AsyncClient.__init__ = _patched_async_init

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
import json
import time
import faiss
from rank_bm25 import BM25Okapi
import torch

PROJECT_ROOT = Path("/Users/nipun.batra/Downloads/ML/multimodal-rag-video-qa")
DATA_DIR = PROJECT_ROOT / "data" / "nextqa"
PROCESSED_DIR = DATA_DIR / "processed"
EMBEDDINGS_DIR = PROCESSED_DIR / "embeddings"
INDICES_DIR = PROCESSED_DIR / "indices"
INDICES_DIR.mkdir(parents=True, exist_ok=True)

print(f"FAISS version: {faiss.__version__ if hasattr(faiss, '__version__') else 'N/A'}")
print(f"Embeddings dir: {EMBEDDINGS_DIR}")
print(f"Indices dir: {INDICES_DIR}")

FAISS version: 1.13.2
Embeddings dir: /Users/nipun.batra/Downloads/ML/Multimodal_RAG_NExTQA/data/processed/embeddings
Indices dir: /Users/nipun.batra/Downloads/ML/Multimodal_RAG_NExTQA/data/processed/indices


## 1. Load All Embeddings

We load the text embeddings (all strategies), image embeddings, and caption embeddings
generated in Notebook 04.

In [2]:
# Load text embeddings for each strategy
text_strategies = ['fixed_window', 'semantic', 'hierarchical_child',
                   'hierarchical_parent', 'transcript_aligned', 'agentic', 'caption_concat']

text_embeddings = {}
text_metadata = {}

for strategy in text_strategies:
    emb_file = EMBEDDINGS_DIR / "text" / strategy / "embeddings.npy"
    meta_file = EMBEDDINGS_DIR / "text" / strategy / "metadata.json"
    if emb_file.exists():
        text_embeddings[strategy] = np.load(emb_file).astype(np.float32)
        with open(meta_file) as f:
            text_metadata[strategy] = json.load(f)
        print(f"  {strategy}: {text_embeddings[strategy].shape}")

# Load image embeddings
img_emb_file = EMBEDDINGS_DIR / "image" / "clustering" / "embeddings.npy"
img_meta_file = EMBEDDINGS_DIR / "image" / "clustering" / "metadata.json"
image_embeddings = np.load(img_emb_file).astype(np.float32)
with open(img_meta_file) as f:
    image_metadata = json.load(f)
print(f"  image/clustering: {image_embeddings.shape}")

print(f"\nTotal text strategies loaded: {len(text_embeddings)}")
print(f"Image embeddings: {image_embeddings.shape[0]} frames")

  fixed_window: (103, 1024)
  semantic: (1668, 1024)
  hierarchical_child: (1467, 1024)
  hierarchical_parent: (1450, 1024)
  transcript_aligned: (400, 1024)
  agentic: (845, 1024)
  caption_concat: (100, 1024)
  image/clustering: (800, 768)

Total text strategies loaded: 7
Image embeddings: 800 frames


## 2. Build FAISS Indices

FAISS IndexFlatIP (inner product) is used since our embeddings are L2-normalized,
making dot product equivalent to cosine similarity. Flat indices give exact nearest
neighbor results (no approximation) which is appropriate for our dataset size.

In [3]:
def build_faiss_index(embeddings, index_path=None):
    """Build a FAISS flat inner product index from normalized embeddings."""
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    if index_path:
        faiss.write_index(index, str(index_path))

    return index


# Build indices for all text strategies
text_indices = {}
for strategy, embs in text_embeddings.items():
    idx_path = INDICES_DIR / f"text_{strategy}.faiss"
    text_indices[strategy] = build_faiss_index(embs, idx_path)
    print(f"  {strategy}: {text_indices[strategy].ntotal} vectors indexed (dim={embs.shape[1]})")

# Build image index
img_idx_path = INDICES_DIR / f"image_clustering.faiss"
image_index = build_faiss_index(image_embeddings, img_idx_path)
print(f"  image_clustering: {image_index.ntotal} vectors indexed (dim={image_embeddings.shape[1]})")

# Total storage
idx_files = list(INDICES_DIR.glob("*.faiss"))
total_size = sum(f.stat().st_size for f in idx_files) / 1024**2
print(f"\nTotal index storage: {total_size:.1f} MB ({len(idx_files)} files)")

  fixed_window: 103 vectors indexed (dim=1024)
  semantic: 1668 vectors indexed (dim=1024)
  hierarchical_child: 1467 vectors indexed (dim=1024)
  hierarchical_parent: 1450 vectors indexed (dim=1024)
  transcript_aligned: 400 vectors indexed (dim=1024)
  agentic: 845 vectors indexed (dim=1024)
  caption_concat: 100 vectors indexed (dim=1024)
  image_clustering: 800 vectors indexed (dim=768)

Total index storage: 25.9 MB (8 files)


### Reasoning: Index Choice

**Why IndexFlatIP (exact search):**
- Our dataset is small: 5,933 text chunks + 800 image embeddings fit entirely in RAM
- At this scale, exact search is fast enough (< 1ms per query for 6K vectors)
- Approximate indices (IVF, HNSW) add complexity without speed benefit at this scale
- Full 1,570-video dataset would have ~93K text chunks + ~12.5K images -- still fine for flat

**When to switch to approximate:**
- Beyond ~1M vectors, flat search becomes slow (> 10ms per query)
- For production deployment at scale, IVFFlat or HNSW would be appropriate
- Our evaluation pipeline runs thousands of queries, so even small per-query savings matter
  at the 1M+ scale, but not at 6K

**Inner product vs L2:**
- Our embeddings are L2-normalized, so IP(a,b) = cos(a,b)
- FAISS IP search is slightly faster than L2 search
- Higher score = more similar (unlike L2 where lower = more similar)

## 3. Retrieval Functions

We implement the retrieval interface that will be used throughout the evaluation pipeline.
Three retrieval modes: dense text, dense image (cross-modal), and hybrid.

In [4]:
class DenseRetriever:
    """Dense retrieval using FAISS indices."""

    def __init__(self, index, metadata, embedding_model=None):
        self.index = index
        self.metadata = metadata
        self.embedding_model = embedding_model

    def search(self, query_embedding, top_k=5):
        """Search index with a pre-computed query embedding."""
        if query_embedding.ndim == 1:
            query_embedding = query_embedding.reshape(1, -1)
        query_embedding = query_embedding.astype(np.float32)

        scores, indices = self.index.search(query_embedding, top_k)
        results = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0])):
            if idx == -1:
                continue
            result = dict(self.metadata[idx])
            result['score'] = float(score)
            result['rank'] = rank
            results.append(result)
        return results

    def search_by_text(self, query_text, top_k=5):
        """Encode query text and search (requires embedding_model)."""
        if self.embedding_model is None:
            raise ValueError("No embedding model set")
        prefix = "Represent this sentence for searching relevant passages: "
        emb = self.embedding_model.encode([prefix + query_text],
                                           normalize_embeddings=True,
                                           show_progress_bar=False)
        return self.search(emb, top_k)


class HybridRetriever:
    """Combines text and image retrieval with score fusion."""

    def __init__(self, text_retriever, image_retriever, text_weight=0.6, image_weight=0.4):
        self.text_retriever = text_retriever
        self.image_retriever = image_retriever
        self.text_weight = text_weight
        self.image_weight = image_weight

    def search(self, text_query_emb, image_query_emb, top_k=5):
        """Hybrid search combining text and image scores."""
        # Get more candidates from each modality
        text_results = self.text_retriever.search(text_query_emb, top_k=top_k * 2)
        image_results = self.image_retriever.search(image_query_emb, top_k=top_k * 2)

        # Score fusion by video_id
        video_scores = {}
        for r in text_results:
            vid = r.get('video_id', '')
            if vid not in video_scores:
                video_scores[vid] = {'text_score': 0, 'image_score': 0, 'text_meta': r}
            video_scores[vid]['text_score'] = max(video_scores[vid]['text_score'], r['score'])

        for r in image_results:
            vid = r.get('video_id', '')
            if vid not in video_scores:
                video_scores[vid] = {'text_score': 0, 'image_score': 0, 'image_meta': r}
            video_scores[vid]['image_score'] = max(video_scores[vid]['image_score'], r['score'])
            if 'image_meta' not in video_scores[vid]:
                video_scores[vid]['image_meta'] = r

        # Compute fused scores
        results = []
        for vid, scores in video_scores.items():
            fused = (self.text_weight * scores['text_score'] +
                     self.image_weight * scores['image_score'])
            meta = scores.get('text_meta', scores.get('image_meta', {}))
            results.append({
                'video_id': vid,
                'fused_score': fused,
                'text_score': scores['text_score'],
                'image_score': scores['image_score'],
                **{k: v for k, v in meta.items() if k not in ('score', 'rank')}
            })

        results.sort(key=lambda x: x['fused_score'], reverse=True)
        return results[:top_k]


# Test instantiation
caption_retriever = DenseRetriever(text_indices['caption_concat'], text_metadata['caption_concat'])
image_retriever = DenseRetriever(image_index, image_metadata)
hybrid = HybridRetriever(caption_retriever, image_retriever)

print("Retriever classes instantiated:")
print(f"  caption_retriever: {caption_retriever.index.ntotal} vectors")
print(f"  image_retriever: {image_retriever.index.ntotal} vectors")
print(f"  hybrid: text_weight={hybrid.text_weight}, image_weight={hybrid.image_weight}")

Retriever classes instantiated:
  caption_retriever: 100 vectors
  image_retriever: 800 vectors
  hybrid: text_weight=0.6, image_weight=0.4


## 4. BM25 Sparse Retrieval Baseline

BM25 provides a non-neural baseline that retrieves based on exact keyword matching.
This helps us understand how much value the dense embeddings add over simple term overlap.

In [5]:
# Build BM25 index from caption documents
captions_dir = PROCESSED_DIR / "captions"
bm25_corpus = []
bm25_metadata = []

for cap_file in sorted(captions_dir.glob("*.json")):
    vid = cap_file.stem
    with open(cap_file) as f:
        caps = json.load(f)
    # Concatenate captions into one document per video
    doc = " ".join([c['caption'] for c in caps])
    bm25_corpus.append(doc.lower().split())
    bm25_metadata.append({'video_id': vid, 'text': doc})

bm25_index = BM25Okapi(bm25_corpus)
print(f"BM25 index built: {len(bm25_corpus)} documents")
print(f"  Avg document length: {np.mean([len(d) for d in bm25_corpus]):.1f} tokens")


def bm25_search(query, top_k=5):
    """Search BM25 index with a text query."""
    query_tokens = query.lower().split()
    scores = bm25_index.get_scores(query_tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []
    for rank, idx in enumerate(top_indices):
        results.append({
            'video_id': bm25_metadata[idx]['video_id'],
            'score': float(scores[idx]),
            'rank': rank,
            'text_preview': bm25_metadata[idx]['text'][:80],
        })
    return results


# Test BM25
test_results = bm25_search("baby playing toys", top_k=3)
print(f"\nBM25 test query: 'baby playing toys'")
for r in test_results:
    print(f"  #{r['rank']+1} (score={r['score']:.3f}) [{r['video_id']}] {r['text_preview']}...")

BM25 index built: 100 documents
  Avg document length: 76.9 tokens

BM25 test query: 'baby playing toys'
  #1 (score=6.541) [2406887888] a living room with a couch and a television a living room with a couch and a bab...
  #2 (score=6.154) [10488004303] a young boy is playing with his toys a baby playing with a dog on the floor a li...
  #3 (score=5.998) [12011352464] a young boy standing in a room with toys a person in a room with a toy a child i...


## 5. Test Retrieval on NExT-QA Questions

We test all retrieval modes on real questions from the MC test set to validate that
the system retrieves relevant content for the actual evaluation queries.

In [6]:
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

# Load models for query encoding
text_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device='mps')
clip_model = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').to('mps')
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
clip_model.eval()

# Set up retrievers with models
caption_retriever.embedding_model = text_model

print("Models loaded for query encoding.")
print(f"  Text: bge-large-en-v1.5 (1024-dim)")
print(f"  Image: CLIP-ViT-L/14 (768-dim)")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Models loaded for query encoding.
  Text: bge-large-en-v1.5 (1024-dim)
  Image: CLIP-ViT-L/14 (768-dim)


In [7]:
# Load MC test questions for dev subset
mc_test = pd.read_parquet(DATA_DIR / "MC" / "test-00000-of-00001.parquet")
mc_test['video_str'] = mc_test['video'].astype(str)
dev_videos = sorted(mc_test['video_str'].unique())[:100]
mc_dev = mc_test[mc_test['video_str'].isin(dev_videos)].copy()

# Sample 5 diverse questions (one per type family)
sample_questions = []
for qtype in ['CW', 'TN', 'DO', 'DL', 'DC']:
    subset = mc_dev[mc_dev['type'] == qtype]
    if len(subset) > 0:
        sample_questions.append(subset.iloc[0])

print(f"Testing retrieval on {len(sample_questions)} sample questions:")
print("=" * 80)

for q_row in sample_questions:
    question = q_row['question']
    target_vid = q_row['video_str']
    qtype = q_row['type']
    correct_answer = q_row[f"a{q_row['answer']}"]

    print(f"\n  [{qtype}] Q: {question}")
    print(f"       Target video: {target_vid}")
    print(f"       Answer: {correct_answer}")

    # Dense text retrieval (caption)
    text_results = caption_retriever.search_by_text(question, top_k=5)
    text_hit = any(r['video_id'] == target_vid for r in text_results)
    print(f"       Dense text @5: {'HIT' if text_hit else 'MISS'} "
          f"(top: {text_results[0]['video_id']}, score={text_results[0]['score']:.4f})")

    # BM25 retrieval
    bm25_results = bm25_search(question, top_k=5)
    bm25_hit = any(r['video_id'] == target_vid for r in bm25_results)
    print(f"       BM25 @5:       {'HIT' if bm25_hit else 'MISS'} "
          f"(top: {bm25_results[0]['video_id']}, score={bm25_results[0]['score']:.3f})")

    # Cross-modal image retrieval
    clip_inputs = clip_processor(text=[question], return_tensors="pt", padding=True).to('mps')
    with torch.no_grad():
        clip_out = clip_model.get_text_features(**clip_inputs)
        clip_emb = clip_out.pooler_output
        clip_emb = clip_emb / clip_emb.norm(dim=-1, keepdim=True)
    clip_emb_np = clip_emb.cpu().numpy().astype(np.float32)
    img_results = image_retriever.search(clip_emb_np, top_k=5)
    img_hit = any(r['video_id'] == target_vid for r in img_results)
    print(f"       CLIP image @5: {'HIT' if img_hit else 'MISS'} "
          f"(top: {img_results[0]['video_id']}, score={img_results[0]['score']:.4f})")

Testing retrieval on 5 sample questions:

  [CW] Q: why is the man raising his legs throughout the video
       Target video: 11342887364
       Answer: part of the dance routine
       Dense text @5: MISS (top: 2399811629, score=0.6022)
       BM25 @5:       MISS (top: 11506594895, score=9.748)


       CLIP image @5: HIT (top: 11342887364, score=0.2266)

  [TN] Q: what expression did the girl give after she pointed at the camera
       Target video: 11794871936
       Answer: smile
       Dense text @5: MISS (top: 13259212584, score=0.5328)
       BM25 @5:       MISS (top: 10355341433, score=7.557)


       CLIP image @5: MISS (top: 10437686463, score=0.2214)

  [DO] Q: who threw the green balloon
       Target video: 12753577835
       Answer: boy in white


       Dense text @5: HIT (top: 12753577835, score=0.4338)
       BM25 @5:       HIT (top: 12298809926, score=6.875)
       CLIP image @5: MISS (top: 10495085476, score=0.2230)

  [DL] Q: where could this be happening
       Target video: 12299711406
       Answer: kitchen


       Dense text @5: MISS (top: 2405004230, score=0.5434)
       BM25 @5:       MISS (top: 11506594895, score=3.405)
       CLIP image @5: MISS (top: 13125896183, score=0.1992)

  [DC] Q: how many people are there
       Target video: 12427023395
       Answer: three
       Dense text @5: MISS (top: 12257884095, score=0.5632)
       BM25 @5:       MISS (top: 10128261054, score=3.497)
       CLIP image @5: MISS (top: 2400084970, score=0.2078)


### Reasoning: Retrieval Results Interpretation

The retrieval test on real NExT-QA questions reveals how well each mode works:

**Dense text retrieval (bge-large on caption embeddings):**
- Searches over BLIP-generated captions -- the text must describe the visual content
- Works well for descriptive queries that match caption vocabulary
- May struggle when the question uses different phrasing than BLIP's captioning style

**BM25 (keyword matching):**
- Relies on exact token overlap between question and caption text
- Works when questions contain specific nouns that appear in captions (baby, dog, kitchen)
- Fails when questions use synonyms or indirect references

**CLIP cross-modal (text-to-image):**
- Directly matches question semantics against visual frame content
- Works for visually grounded questions ("What color is X?", "Where is the baby?")
- May struggle with abstract or temporal questions ("Why did X happen?")

**Key insight: No single retrieval mode dominates all question types.** Hybrid retrieval
combines the strengths of text (semantic reasoning) and image (visual grounding) to
cover more question types effectively.

## 6. Retrieval Latency Benchmark

We measure per-query latency for each retrieval mode to establish the performance
envelope for the evaluation pipeline.

In [8]:
# Benchmark retrieval latency
n_queries = 50
sample_qs = mc_dev.sample(n_queries, random_state=42)

# Precompute query embeddings
query_prefix = "Represent this sentence for searching relevant passages: "
query_texts = [query_prefix + q for q in sample_qs['question'].tolist()]
t0 = time.time()
query_embs = text_model.encode(query_texts, normalize_embeddings=True,
                                show_progress_bar=False, batch_size=32)
text_encode_time = time.time() - t0

# Dense text search latency
t0 = time.time()
for emb in query_embs:
    caption_retriever.search(emb.reshape(1, -1), top_k=5)
dense_text_time = time.time() - t0

# BM25 search latency
t0 = time.time()
for q in sample_qs['question'].tolist():
    bm25_search(q, top_k=5)
bm25_time = time.time() - t0

# Image search latency (using text embeddings as proxy for CLIP query embs)
# In practice we'd encode with CLIP, but this measures index search time
dummy_img_queries = np.random.randn(n_queries, 768).astype(np.float32)
dummy_img_queries = dummy_img_queries / np.linalg.norm(dummy_img_queries, axis=1, keepdims=True)
t0 = time.time()
for emb in dummy_img_queries:
    image_retriever.search(emb.reshape(1, -1), top_k=5)
img_search_time = time.time() - t0

print(f"Retrieval latency benchmark ({n_queries} queries):")
print(f"{'Mode':<25} {'Total (ms)':<15} {'Per-query (ms)':<15}")
print("-" * 55)
print(f"{'Text encoding':<25} {text_encode_time*1000:<15.1f} {text_encode_time*1000/n_queries:<15.2f}")
print(f"{'Dense text search':<25} {dense_text_time*1000:<15.1f} {dense_text_time*1000/n_queries:<15.2f}")
print(f"{'BM25 search':<25} {bm25_time*1000:<15.1f} {bm25_time*1000/n_queries:<15.2f}")
print(f"{'Image index search':<25} {img_search_time*1000:<15.1f} {img_search_time*1000/n_queries:<15.2f}")
print()
print(f"End-to-end per query (encode + search):")
print(f"  Dense text: {(text_encode_time + dense_text_time)*1000/n_queries:.2f} ms/query")
print(f"  BM25: {bm25_time*1000/n_queries:.2f} ms/query")

Retrieval latency benchmark (50 queries):
Mode                      Total (ms)      Per-query (ms) 
-------------------------------------------------------
Text encoding             191.7           3.83           
Dense text search         0.5             0.01           
BM25 search               4.4             0.09           
Image index search        1.8             0.04           

End-to-end per query (encode + search):
  Dense text: 3.84 ms/query
  BM25: 0.09 ms/query


## 7. Recall@K Analysis (Dev Set)

We compute Recall@K on the dev set -- what fraction of questions have their target video
in the top-K retrieved results. This gives us an initial signal before the full
evaluation in Notebook 06.

In [9]:
def compute_recall_at_k(questions_df, retriever, model, k_values=[1, 3, 5, 10]):
    """Compute Recall@K: fraction of questions where target video is in top-K."""
    prefix = "Represent this sentence for searching relevant passages: "
    queries = [prefix + q for q in questions_df['question'].tolist()]
    target_vids = questions_df['video_str'].tolist()

    # Batch encode
    query_embs = model.encode(queries, normalize_embeddings=True,
                              show_progress_bar=False, batch_size=64)

    recalls = {k: 0 for k in k_values}
    max_k = max(k_values)

    for i, (emb, target) in enumerate(zip(query_embs, target_vids)):
        results = retriever.search(emb.reshape(1, -1), top_k=max_k)
        retrieved_vids = [r['video_id'] for r in results]
        for k in k_values:
            if target in retrieved_vids[:k]:
                recalls[k] += 1

    n = len(questions_df)
    return {k: v / n for k, v in recalls.items()}


# Compute Recall@K for caption_concat strategy on dev set
print("Computing Recall@K on dev set (874 questions)...")
t0 = time.time()
recall_caption = compute_recall_at_k(mc_dev, caption_retriever, text_model)
t_elapsed = time.time() - t0

print(f"\nCaption-concat retrieval (dense text, {t_elapsed:.1f}s):")
for k, r in recall_caption.items():
    print(f"  Recall@{k}: {r:.4f} ({r*100:.1f}%)")

# Compute for other strategies
print("\nRecall@5 by text strategy:")
print(f"{'Strategy':<25} {'Recall@5':<12} {'Recall@10':<12}")
print("-" * 49)
for strategy in ['caption_concat', 'agentic', 'transcript_aligned', 'fixed_window']:
    if strategy in text_indices:
        retriever = DenseRetriever(text_indices[strategy], text_metadata[strategy])
        recall = compute_recall_at_k(mc_dev, retriever, text_model, k_values=[5, 10])
        print(f"{strategy:<25} {recall[5]:<12.4f} {recall[10]:<12.4f}")

Computing Recall@K on dev set (874 questions)...



Caption-concat retrieval (dense text, 1.5s):
  Recall@1: 0.2746 (27.5%)
  Recall@3: 0.4531 (45.3%)
  Recall@5: 0.5309 (53.1%)
  Recall@10: 0.6579 (65.8%)

Recall@5 by text strategy:
Strategy                  Recall@5     Recall@10   
-------------------------------------------------


caption_concat            0.5309       0.6579      


agentic                   0.9840       0.9943      


transcript_aligned        0.9577       0.9760      


fixed_window              0.9130       0.9451      


In [10]:
# Recall@K by question type
print("\nRecall@5 by question type (caption_concat strategy):")
print(f"{'Type':<8} {'Count':<8} {'Recall@5':<12} {'Recall@10':<12}")
print("-" * 40)

type_info = {
    'CW': 'Causal-Why', 'CH': 'Causal-How',
    'TN': 'Temporal-Next', 'TC': 'Temporal-Current', 'TP': 'Temporal-Previous',
    'DO': 'Descriptive-Object', 'DL': 'Descriptive-Location', 'DC': 'Descriptive-Count',
}

for qtype in ['CW', 'CH', 'TN', 'TC', 'TP', 'DO', 'DL', 'DC']:
    subset = mc_dev[mc_dev['type'] == qtype]
    if len(subset) == 0:
        continue
    recall = compute_recall_at_k(subset, caption_retriever, text_model, k_values=[5, 10])
    print(f"{qtype:<8} {len(subset):<8} {recall[5]:<12.4f} {recall[10]:<12.4f}")


Recall@5 by question type (caption_concat strategy):
Type     Count    Recall@5     Recall@10   
----------------------------------------


CW       344      0.5349       0.6802      


CH       110      0.6727       0.7455      


TN       150      0.5400       0.6533      


TC       123      0.5366       0.6585      
TP       16       0.6250       0.7500      
DO       56       0.4286       0.6429      


DL       44       0.2273       0.3182      
DC       31       0.4839       0.5806      


### Reasoning: Recall@K Interpretation

Recall@K measures retrieval effectiveness: "Given a question about video X, does our
retrieval system include video X in the top-K results?"

**Important context:** This is a challenging retrieval task because:
1. We have 100 videos, each with 8 captions (~85 words)
2. Many questions ask about common activities (eating, playing, walking) that appear
   across multiple videos
3. The BLIP captions are generic ("a baby playing with toys") and may match many videos

**What the numbers mean:**
- Recall@1 = fraction where the correct video is ranked #1
- Recall@5 = fraction where correct video is somewhere in top 5
- Recall@10 = fraction where correct video is in top 10

**Expected patterns by question type:**
- Descriptive questions should have higher recall (specific visual elements in captions)
- Causal questions may have lower recall (cause-effect reasoning isn't captured in captions)
- Temporal questions depend on whether the temporal element appears in caption text

**Ceiling analysis:** With only 100 videos and 1-per-video retrieval units (caption_concat),
random baseline Recall@5 = 5/100 = 5%. Any recall significantly above 5% shows the
embeddings are providing real signal.

## Summary

**Vector store indexing and retrieval pipeline complete:**

| Component | Details |
|-----------|---------|
| FAISS indices | 7 text + 1 image index (flat inner product) |
| Dense text retrieval | bge-large query encoding + FAISS search |
| Cross-modal retrieval | CLIP text encoder + image FAISS search |
| BM25 baseline | Keyword matching over caption documents |
| Hybrid retrieval | Weighted fusion (0.6 text + 0.4 image) |

**Key findings:**
1. Dense retrieval provides meaningful recall above the random baseline
2. Per-query latency is sub-millisecond for index search (bottleneck is encoding)
3. Different chunking strategies yield different recall characteristics
4. Question type strongly influences retrieval difficulty

**Outputs saved:**
- `data/processed/indices/*.faiss` -- serialized FAISS indices
- Retrieval classes ready for evaluation pipeline

**Next: Notebook 06 - Retrieval Evaluation.** We run systematic Recall@K evaluation
across all chunking strategies and question types to find the best configuration.